# AURION — Rendimiento por escenario de despliegue

El test set contiene dos tipos de imagen muy distintos:

- **Escena de control** — uno o dos palets, grandes, plano completo. Es lo que verá
  la cámara fija del puesto de inspección. **Es el caso de uso real.**
- **Escena de almacén** — pasillos y estanterías con veinte o treinta palets a
  distintas profundidades, muchos diminutos o parcialmente ocultos. Fuera de
  especificación.

Mezclarlas da un único mAP que no describe bien ninguno de los dos casos. Los
fallos críticos se concentran en las escenas densas, así que la métrica global
es **pesimista** para el despliegue previsto.

Este notebook las separa y evalúa por separado.

> Esto no es maquillar el número. Es reportar el rendimiento dentro y fuera del
> dominio de diseño, que es más informativo que un promedio de los dos.

## 0. Preparación de la sesión

Colab borra `/content` al reiniciar. Esta sección lo reconstruye desde cero.

**Sube estos dos archivos** con el icono de carpeta del panel izquierdo, o móntalos
desde Drive con la celda siguiente:

- `aurion_split.zip`
- `best.pt`

In [ ]:
!pip install -q ultralytics
import ultralytics; print("ultralytics", ultralytics.__version__)

### Opción A — desde Drive (recomendado)

Si guardaste los archivos en Drive, ajusta las rutas. Si los subiste a mano al
panel izquierdo, **salta esta celda**.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/AURION/aurion_split.zip" /content/
# !cp "/content/drive/MyDrive/AURION/best.pt" /content/
print("Salta esta celda si subiste los archivos a mano")

### Reconstruir el split

Busca sola dónde quedó `test/images` dentro del zip, así que da igual si trae
carpeta contenedora o no.

In [ ]:
from pathlib import Path
import shutil

ZIP    = Path("/content/aurion_split.zip")
PESOS  = Path("/content/best.pt")
SPLIT  = Path("/content/aurion_split_ok")

CLASES = [
    "palet_bueno",
    "palet_roto",
    "paquete_emb_correct_dim_correct",
    "paquete_emb_correct_dim_incorrect",
    "paquete_emb_incorrect_dim_correct",
    "paquete_emb_incorrect_dim_incorrect",
]

assert ZIP.exists(),   f"Falta {ZIP} — súbelo al panel izquierdo"
assert PESOS.exists(), f"Falta {PESOS} — súbelo al panel izquierdo"

if not (SPLIT/"test"/"images").is_dir():
    tmp = Path("/content/_tmp_unzip")
    if tmp.exists(): shutil.rmtree(tmp)
    shutil.unpack_archive(str(ZIP), str(tmp))

    # localizar la raiz real del dataset dentro del zip
    base = next(p.parent.parent for p in tmp.rglob("test/images"))
    SPLIT.mkdir(exist_ok=True)
    for d in ["train", "valid", "test"]:
        if (base/d).is_dir() and not (SPLIT/d).exists():
            shutil.move(str(base/d), str(SPLIT/d))
    shutil.rmtree(tmp, ignore_errors=True)
    print("Split reconstruido")
else:
    print("El split ya existía")

# data.yaml con rutas de Colab (el original lleva rutas de Windows)
(SPLIT/"data.yaml").write_text(
    f"path: {SPLIT.as_posix()}\ntrain: train/images\nval: valid/images\n"
    f"test: test/images\n\nnc: {len(CLASES)}\nnames: {CLASES}\n")

for d in ["train", "valid", "test"]:
    n = len(list((SPLIT/d/"images").iterdir())) if (SPLIT/d/"images").is_dir() else 0
    print(f"  {d:<6} {n:>5} imágenes")
print("\nEn test deberían salir 127.")

---
## 1. Configuración

In [ ]:
from pathlib import Path
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# PESOS, SPLIT y CLASES vienen de la seccion 0
OUT   = Path("/content/analisis")
IMGSZ = 640
EXT_IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

OUT.mkdir(parents=True, exist_ok=True)
assert PESOS.exists(), f"No existe {PESOS}"
assert (SPLIT/"test"/"images").is_dir(), f"No existe {SPLIT}/test/images"

model = YOLO(str(PESOS))
print("OK")

---
## 2. Caracterizar cada imagen

Dos medidas por imagen, ambas de la **verdad terreno** (no de las predicciones,
para que el criterio no dependa del modelo):

- `n_obj` — cuántos objetos anotados hay
- `area_max` — área del objeto mayor, como fracción de la imagen

In [ ]:
filas = []
for f in sorted((SPLIT/"test"/"images").iterdir()):
    if f.suffix.lower() not in EXT_IMG: continue
    txt = SPLIT/"test"/"labels"/(f.stem + ".txt")
    areas, clases = [], []
    if txt.exists():
        for l in txt.read_text().splitlines():
            p = l.split()
            if len(p) < 5: continue
            clases.append(int(p[0]))
            areas.append(float(p[3]) * float(p[4]))   # bw*bh normalizados
    filas.append({
        "imagen": f.name,
        "n_obj": len(areas),
        "area_max": round(max(areas), 4) if areas else 0.0,
        "area_media": round(float(np.mean(areas)), 4) if areas else 0.0,
        "tiene_roto": 1 in clases,
    })

car = pd.DataFrame(filas)
print(f"{len(car)} imágenes de test\n")
print(car[["n_obj","area_max","area_media"]].describe().round(4).to_string())

### Distribución

Si hay dos poblaciones distintas, se verá aquí.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(car.n_obj, bins=range(0, int(car.n_obj.max())+2), color="steelblue", edgecolor="w")
ax[0].set_title("objetos por imagen"); ax[0].set_xlabel("n_obj"); ax[0].grid(alpha=.3)
ax[1].hist(car.area_max, bins=25, color="darkorange", edgecolor="w")
ax[1].set_title("área del objeto mayor"); ax[1].set_xlabel("fracción de imagen"); ax[1].grid(alpha=.3)
ax[2].scatter(car.n_obj, car.area_max, alpha=.6, c=car.tiene_roto.map({True:"crimson",False:"steelblue"}))
ax[2].set_xlabel("n_obj"); ax[2].set_ylabel("area_max")
ax[2].set_title("rojo = contiene palet_roto"); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

print("Cuantiles de n_obj:")
print(car.n_obj.quantile([.25,.5,.75,.9]).to_string())

### Criterio

Ajusta los dos umbrales según lo que hayas visto arriba. Los valores por defecto
son un punto de partida razonable: un puesto de control ve un palet con su carga,
o sea pocos objetos y grandes.

In [ ]:
MAX_OBJ  = 6      # como mucho este número de objetos
MIN_AREA = 0.04   # el mayor ocupa al menos este porcentaje de la imagen

car["escenario"] = np.where(
    (car.n_obj <= MAX_OBJ) & (car.area_max >= MIN_AREA),
    "control", "almacen")

print(car.escenario.value_counts().to_string())
print()
print(car.groupby("escenario")[["n_obj","area_max"]].mean().round(3).to_string())

n_ctrl = (car.escenario == "control").sum()
if n_ctrl < 30:
    print(f"\nAVISO: solo {n_ctrl} imágenes de control. La métrica será ruidosa.")
    print("Relaja MAX_OBJ o baja MIN_AREA.")

### Verificación visual

**Míralo antes de seguir.** Si la clasificación no se corresponde con lo que
esperabas, ajusta los umbrales y repite. El criterio tiene que ser defendible,
no conveniente.

In [ ]:
for esc in ["control", "almacen"]:
    sub = car[car.escenario == esc].head(6)
    if not len(sub): continue
    fig, axes = plt.subplots(1, len(sub), figsize=(3.2*len(sub), 3.4))
    for ax, (_, r) in zip(np.atleast_1d(axes), sub.iterrows()):
        im = cv2.imread(str(SPLIT/"test"/"images"/r.imagen))
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(f"n={r.n_obj}  a={r.area_max:.3f}", fontsize=9); ax.axis("off")
    fig.suptitle(f"ESCENA DE {esc.upper()}", fontsize=13)
    plt.tight_layout(); plt.show()

---
## 3. Evaluar por separado

In [ ]:
def construir(nombres, dst):
    if dst.exists(): shutil.rmtree(dst)
    (dst/"test"/"images").mkdir(parents=True)
    (dst/"test"/"labels").mkdir(parents=True)
    for n in nombres:
        shutil.copy2(SPLIT/"test"/"images"/n, dst/"test"/"images"/n)
        txt = SPLIT/"test"/"labels"/(Path(n).stem + ".txt")
        if txt.exists(): shutil.copy2(txt, dst/"test"/"labels"/txt.name)
    (dst/"data.yaml").write_text(
        f"path: {dst.as_posix()}\ntrain: test/images\nval: test/images\n"
        f"test: test/images\n\nnc: {len(CLASES)}\nnames: {CLASES}\n")
    return dst/"data.yaml"

res = []
for esc in ["control", "almacen"]:
    nombres = car[car.escenario == esc].imagen.tolist()
    if not nombres: continue
    yml = construir(nombres, Path(f"/content/_esc_{esc}"))
    r = model.val(data=str(yml), split="test", imgsz=IMGSZ, verbose=False, plots=False)
    fila = {"escenario": esc, "imagenes": len(nombres),
            "mAP50": round(float(r.box.map50), 4),
            "mAP50_95": round(float(r.box.map), 4),
            "precision": round(float(r.box.mp), 4),
            "recall": round(float(r.box.mr), 4),
            "recall_palet_roto": round(float(r.box.r[1]), 4)}
    for i, c in enumerate(CLASES):
        fila[f"mAP_{c}"] = round(float(r.box.maps[i]), 4)
    res.append(fila)
    print(f"{esc}: {len(nombres)} imágenes  mAP50 {fila['mAP50']:.4f}  "
          f"mAP50-95 {fila['mAP50_95']:.4f}")

# Global, para comparar
rg = model.val(data=str(SPLIT/"data.yaml"), split="test", imgsz=IMGSZ,
               verbose=False, plots=False)
res.append({"escenario": "TODO", "imagenes": len(car),
            "mAP50": round(float(rg.box.map50), 4),
            "mAP50_95": round(float(rg.box.map), 4),
            "precision": round(float(rg.box.mp), 4),
            "recall": round(float(rg.box.mr), 4),
            "recall_palet_roto": round(float(rg.box.r[1]), 4),
            **{f"mAP_{c}": round(float(rg.box.maps[i]), 4) for i, c in enumerate(CLASES)}})

esc_df = pd.DataFrame(res)
esc_df.to_csv(OUT/"por_escenario.csv", index=False)
car.to_csv(OUT/"caracterizacion_test.csv", index=False)

### La tabla que va a la web

In [ ]:
cols = ["escenario","imagenes","mAP50","mAP50_95","precision","recall","recall_palet_roto"]
print(esc_df[cols].to_string(index=False))
print()

t = esc_df.set_index("escenario")
if "control" in t.index and "TODO" in t.index:
    d = t.loc["control","mAP50_95"] - t.loc["TODO","mAP50_95"]
    print(f"Dentro de especificación vs global: {d:+.4f} de mAP50-95 "
          f"({100*d/t.loc['TODO','mAP50_95']:+.1f}%)")
if "control" in t.index and "almacen" in t.index:
    print(f"Control vs almacén: {t.loc['control','mAP50_95'] - t.loc['almacen','mAP50_95']:+.4f}")

print("\nPor clase:")
pc = esc_df.set_index("escenario")[[f"mAP_{c}" for c in CLASES]].T
pc.index = [i.replace("mAP_","") for i in pc.index]
print(pc.to_string())

In [ ]:
x = np.arange(len(CLASES)); w = 0.35
fig, ax = plt.subplots(figsize=(13, 5))
for i, esc in enumerate([e for e in ["control","almacen"] if e in esc_df.escenario.values]):
    v = [esc_df[esc_df.escenario==esc][f"mAP_{c}"].iloc[0] for c in CLASES]
    ax.bar(x + i*w, v, w, label=f"escena de {esc}")
ax.set_xticks(x + w/2)
ax.set_xticklabels([c.replace("paquete_","").replace("_"," ") for c in CLASES],
                   rotation=20, ha="right", fontsize=9)
ax.set_ylabel("mAP50-95"); ax.set_ylim(0, 1); ax.grid(alpha=.3, axis="y"); ax.legend()
ax.set_title("Rendimiento por clase, dentro y fuera del dominio de diseño")
plt.tight_layout(); plt.savefig(OUT/"por_escenario.png", dpi=150); plt.show()

---
## 4. Cómo contarlo

Si el número de la escena de control sale bastante por encima del global, la web
debe reportar **los dos**, con esta lectura:

> El sistema se diseña para un puesto de control con cámara fija donde pasa un
> palet con su carga. Dentro de ese dominio alcanza mAP50-95 de **X**. Evaluado
> también sobre escenas de almacén general (estanterías con decenas de palets a
> distintas profundidades), que quedan fuera de especificación, el rendimiento
> baja a **Y**. Se reportan ambos para delimitar el dominio de validez.

Eso es más informativo que un promedio, y sobre todo es **verificable**: el
criterio de separación está en el código y cualquiera puede reproducirlo.

Lo que no vale es quedarse solo con el número bueno y no mencionar el otro.

In [ ]:
shutil.make_archive("/content/aurion_escenarios", "zip", OUT)
from google.colab import files
files.download("/content/aurion_escenarios.zip")